# Notebook 01 — Data Exploration and Preprocessing

**MALDI-MSI Analysis of Mouse Urinary Bladder**  
Author: Reza Rajaee

---

## What this notebook covers

1. What is MALDI-MSI data and how is it structured
2. Loading and exploring the dataset
3. Ion images and pixel spectra
4. Complete preprocessing pipeline
5. Identifying top peaks and their molecular identity
6. PCA and t-SNE — before and after preprocessing

---

## Dataset

**Mouse urinary bladder** — Römpp et al. (2010) *Angew. Chem. Int. Ed.* 49:3834  
**Source:** PRIDE accession PXD001283  
**Instrument:** AP-SMALDI LTQ Orbitrap (10 µm spatial resolution)  
**Data format:** Centroided imzML — peaks pre-detected by instrument

The bladder tissue contains three anatomical layers:
- **Urothelium** — inner epithelial lining
- **Muscle layer** — smooth muscle tissue
- **Connective tissue** — outer supportive layer

These layers have distinct lipid compositions, making this dataset ideal
for evaluating molecular segmentation methods.


## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from pathlib import Path
import sys, os

os.chdir("/workspaces/maldi-msi-analysis")
sys.path.insert(0, "/workspaces/maldi-msi-analysis")

os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables",  exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "font.size": 10,
                      "axes.spines.top": False,
                      "axes.spines.right": False})
print("Setup complete.")
print("Working directory:", os.getcwd())


## 1. What is MALDI-MSI Data?

MALDI-MSI (Matrix-Assisted Laser Desorption/Ionization Mass Spectrometry Imaging)
is a technique that measures the molecular composition of tissue at spatial resolution.

**How it works:**
1. A thin tissue section is placed on a conductive slide
2. A chemical matrix is sprayed onto the tissue
3. A UV laser scans the tissue pixel by pixel
4. At each pixel, the laser ionizes molecules which are then separated by mass
5. The detector records which molecules are present and at what abundance

**The result is a 3D data cube:**
```
pixels (x) × pixels (y) × m/z values
```

At each pixel you get a **mass spectrum** — a list of detected molecules
(identified by their mass-to-charge ratio, m/z) and their intensities.

Each molecule has a characteristic m/z value:
- Phosphatidylcholine (PC) lipids: typically 700-850 Da
- Lysophospholipids: typically 500-600 Da
- Sphingomyelins: typically 700-830 Da

Plotting the intensity of one m/z value across all pixels gives an
**ion image** — the spatial distribution of that molecule in the tissue.


## 2. Load the Dataset

We load the imzML file and restrict to the 400-1000 Da mass range,
which covers the main lipid classes present in tissue.

> **Note:** Run the download cell in `data/README.md` first if you
> have not downloaded the dataset yet.


In [ ]:
from src.load import load_imzml

data = load_imzml(
    filepath = "data/mouse_bladder.imzML",
    mz_min   = 400,
    mz_max   = 1000,
    verbose  = True
)

spectra     = data["spectra"]
mz_values   = data["mz_values"]
coordinates = data["coordinates"]

print(f"\nDataset summary:")
print(f"  Pixels:    {data['n_pixels']}")
print(f"  m/z range: {mz_values.min():.2f} — {mz_values.max():.2f} Da")
print(f"  m/z bins:  {data['n_mz']}")
print(f"  Image dimensions: x={coordinates['x'].max()} y={coordinates['y'].max()}")


## 3. Total Ion Current (TIC) Map

The TIC map shows the sum of all peak intensities at each pixel.
It gives a spatial overview of where the tissue is located and
where signal is strongest. Bright areas = more total molecular signal.


In [ ]:
from src.load import reconstruct_image

tic     = spectra.sum(axis=1)
tic_map = reconstruct_image(tic, coordinates)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(tic_map, cmap="viridis", interpolation="nearest", aspect="equal")
plt.colorbar(im, ax=ax, label="Total Ion Current")
ax.set_title("TIC Map — Mouse Urinary Bladder\nSpatial overview of molecular signal")
ax.set_xlabel("x (pixels)")
ax.set_ylabel("y (pixels)")
plt.tight_layout()
plt.savefig("results/figures/01_tic_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


## 4. Mean Spectrum

The mean spectrum averages all pixel spectra together.
It shows which m/z values (molecules) are detected across the tissue
and at what relative abundance. Prominent peaks correspond to
abundant lipid species in bladder tissue.


In [ ]:
mean_spectrum = spectra.mean(axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mz_values, mean_spectrum, linewidth=0.5, color="#2d6a9f", alpha=0.9)
ax.set_xlabel("m/z (Da)")
ax.set_ylabel("Mean intensity")
ax.set_title("Mean spectrum — average of all pixels\n"
             "Each peak represents one molecular species")
plt.tight_layout()
plt.savefig("results/figures/01_mean_spectrum.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


## 5. Individual Ion Images

An **ion image** shows where one specific molecule is located in the tissue.
We plot the 9 most abundant peaks to explore molecular heterogeneity.

Different molecules concentrate in different tissue layers —
this is the basis for molecular segmentation.


In [ ]:
top9_idx = np.argsort(mean_spectrum)[-9:][::-1]
top9_mz  = mz_values[top9_idx]

print("Top 9 most abundant m/z values:")
for i, (idx, mz) in enumerate(zip(top9_idx, top9_mz)):
    print(f"  {i+1}. m/z = {mz:.4f}  (mean intensity = {mean_spectrum[idx]:.6f})")

fig, axes = plt.subplots(3, 3, figsize=(13, 11))
axes = axes.flatten()

x = coordinates["x"].values.astype(int) - coordinates["x"].min()
y = coordinates["y"].values.astype(int) - coordinates["y"].min()

for ax, idx, mz in zip(axes, top9_idx, top9_mz):
    ion_map = np.full((y.max()+1, x.max()+1), np.nan)
    ion_map[y, x] = spectra[:, idx]
    im = ax.imshow(ion_map, cmap="viridis",
                   interpolation="nearest", aspect="equal")
    plt.colorbar(im, ax=ax, shrink=0.7)
    ax.set_title(f"m/z = {mz:.4f}", fontsize=9)
    ax.set_xlabel("x", fontsize=8)
    ax.set_ylabel("y", fontsize=8)

fig.suptitle("Ion images — top 9 most abundant molecular features\n"
             "Each image shows the spatial distribution of one molecule",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/01_ion_images_top9.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


## 6. Comparing Individual Pixel Spectra

Each pixel has its own spectrum. Pixels from different tissue regions
have different spectral profiles — this is what makes spatial segmentation possible.

We compare one pixel from the tissue centre vs one from the periphery
to illustrate molecular heterogeneity.


In [ ]:
# Find pixels roughly at centre and periphery
cx = coordinates["x"].mean()
cy = coordinates["y"].mean()

dist_from_centre = np.sqrt(
    (coordinates["x"] - cx)**2 + (coordinates["y"] - cy)**2
)

# Centre pixel — smallest distance
centre_idx = dist_from_centre.idxmin()
# Periphery pixel — largest distance
periph_idx = dist_from_centre.idxmax()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for ax, idx, label, colour in zip(
    axes,
    [centre_idx, periph_idx],
    ["Centre pixel (tissue core)", "Periphery pixel (tissue edge)"],
    ["#2d6a9f", "#e07b39"]
):
    ax.vlines(mz_values, ymin=0, ymax=spectra[idx],
              linewidth=0.5, color=colour, alpha=0.8)
    ax.set_ylabel("Intensity")
    ax.set_title(f"{label}  "
                 f"(x={coordinates['x'][idx]}, y={coordinates['y'][idx]})",
                 fontsize=10)

axes[-1].set_xlabel("m/z (Da)")
fig.suptitle("Pixel spectra comparison\n"
             "Different locations in tissue have different molecular profiles",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/01_pixel_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


## 7. Preprocessing Pipeline

The data is already centroided (peaks pre-detected by the instrument,
baseline removed). We apply three preprocessing steps:

### Step 1 — TIC Normalisation
Divides each spectrum by its total intensity. Corrects for
pixel-to-pixel variation in total signal caused by differences
in tissue thickness, matrix crystal size, or laser absorption.
Reference: Deininger et al. (2011) *Anal. Chem.* 84:1277

### Step 2 — Peak Frequency Filtering (2.5%)
Keeps only peaks present in at least 2.5% of all pixels.
Removes noise peaks that appear in very few pixels and are
unlikely to represent real molecular signals.
Reference: Bemis, Föll et al. (2023) *Nature Methods* 20:1883

### Step 3 — Rebinning at 10 ppm
Even in centroided data, the same molecule may be detected at
slightly different m/z positions in different pixels due to
instrument calibration variation. Rebinning aligns all pixels
to a common reference m/z grid using 10 ppm tolerance —
matching the Orbitrap mass accuracy of this instrument.
Reference: Bemis, Föll et al. (2023) *Nature Methods* 20:1883


In [ ]:
from src.preprocessing import run_preprocessing

preprocessed = run_preprocessing(
    spectra       = spectra,
    mz_values     = mz_values,
    min_freq      = 0.025,
    tolerance_ppm = 10.0,
    verbose       = True
)

spectra_norm = preprocessed["spectra_norm"]
spectra_pp   = preprocessed["spectra_preprocessed"]
ref_mz       = preprocessed["reference_mz"]

print(f"\nPreprocessing result:")
print(f"  Original features: {spectra.shape[1]}")
print(f"  After filtering:   {len(ref_mz)}")
print(f"  Final matrix:      {spectra_pp.shape}")


### Visualise preprocessing effect — mean spectrum before and after

In [ ]:
mean_before = spectra.mean(axis=0)
mean_after  = spectra_pp.mean(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(mz_values, mean_before, linewidth=0.5,
             color="#888888", alpha=0.9)
axes[0].set_title(f"Before preprocessing — {len(mz_values)} m/z values",
                  fontsize=10)
axes[0].set_ylabel("Mean intensity")

axes[1].vlines(ref_mz, ymin=0, ymax=mean_after,
               linewidth=0.8, color="#2d6a9f", alpha=0.8)
axes[1].set_title(f"After preprocessing — {len(ref_mz)} reference peaks",
                  fontsize=10)
axes[1].set_ylabel("Mean intensity (TIC normalised)")
axes[1].set_xlabel("m/z (Da)")

fig.suptitle("Effect of preprocessing on mean spectrum",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/01_preprocessing_effect.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


### TIC map after normalisation

In [ ]:
tic_norm     = spectra_norm.sum(axis=1)
tic_norm_map = reconstruct_image(tic_norm, coordinates)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im0 = axes[0].imshow(tic_map, cmap="viridis",
                      interpolation="nearest", aspect="equal")
plt.colorbar(im0, ax=axes[0], label="TIC", shrink=0.8)
axes[0].set_title("Before normalisation")

im1 = axes[1].imshow(tic_norm_map, cmap="viridis",
                      interpolation="nearest", aspect="equal")
plt.colorbar(im1, ax=axes[1], label="TIC (normalised)", shrink=0.8)
axes[1].set_title("After TIC normalisation")

for ax in axes:
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")

fig.suptitle("TIC map before and after normalisation\n"
             "Normalisation reduces pixel-level technical variation",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/01_tic_normalisation.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


## 8. Top 5 Peaks — Molecular Identity

The most abundant peaks in the preprocessed data correspond to
known lipid species. In the 400-1000 Da range, MALDI-MSI of
tissue predominantly detects phospholipids.

Common phospholipid classes in bladder tissue:
- **PC (Phosphatidylcholine)**: ~700-850 Da, abundant in cell membranes
- **SM (Sphingomyelin)**: ~700-830 Da, enriched in myelin
- **PE (Phosphatidylethanolamine)**: ~650-800 Da
- **LPC (Lysophosphatidylcholine)**: ~500-600 Da

We show the top 5 peaks by mean intensity and their ion images.

> **Note on identification:** Full lipid identification requires
> MS/MS fragmentation data. Here we report the m/z values and
> probable lipid class assignments based on mass ranges.
> Reference: Hsu & Turk (2009) *J. Chromatogr. B* 877:2714


In [ ]:
mean_pp  = spectra_pp.mean(axis=0)
top5_idx = np.argsort(mean_pp)[-5:][::-1]
top5_mz  = ref_mz[top5_idx]

# Lipid class assignment based on m/z range
def assign_lipid_class(mz):
    if 480 <= mz <= 560:  return "LPC (Lysophosphatidylcholine)"
    elif 650 <= mz <= 760: return "PE (Phosphatidylethanolamine)"
    elif 700 <= mz <= 830: return "PC (Phosphatidylcholine) / SM (Sphingomyelin)"
    elif 830 <= mz <= 900: return "PC (Phosphatidylcholine)"
    else:                   return "Unknown lipid class"

print("Top 5 most abundant peaks after preprocessing:")
print(f"{'Rank':<6} {'m/z (Da)':<12} {'Mean intensity':<18} {'Probable class'}")
print("-" * 70)
for rank, (idx, mz) in enumerate(zip(top5_idx, top5_mz), 1):
    lipid = assign_lipid_class(mz)
    print(f"{rank:<6} {mz:<12.4f} {mean_pp[idx]:<18.6f} {lipid}")


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))

for ax, idx, mz in zip(axes, top5_idx, top5_mz):
    ion_map = np.full((y.max()+1, x.max()+1), np.nan)
    ion_map[y, x] = spectra_pp[:, idx]
    im = ax.imshow(ion_map, cmap="viridis",
                   interpolation="nearest", aspect="equal")
    plt.colorbar(im, ax=ax, shrink=0.7)
    lipid_short = assign_lipid_class(mz).split("(")[0].strip()
    ax.set_title(f"m/z {mz:.2f}\n{lipid_short}", fontsize=8)
    ax.set_xlabel("x", fontsize=7)
    ax.set_ylabel("y", fontsize=7)

fig.suptitle("Ion images of top 5 most abundant peaks after preprocessing",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/01_top5_ion_images.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


## 9. PCA — Before and After Preprocessing

PCA (Principal Component Analysis) reduces the high-dimensional
pixel data to 2D for visualisation. Each point is one pixel.

We run PCA both before and after preprocessing to show:
- Before: pixels may not separate clearly due to technical variation
- After: molecular differences between tissue regions become visible

If the tissue has distinct molecular regions, we expect to see
clusters of pixels in PCA space even before applying any
segmentation algorithm.

Reference: Alexandrov et al. (2010) *J. Proteome Res.* 9:6535


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# PCA before preprocessing (on raw normalised data)
scaler_before = StandardScaler()
X_before      = scaler_before.fit_transform(spectra_norm)
pca_before    = PCA(n_components=2, random_state=42)
pc_before     = pca_before.fit_transform(X_before)

# PCA after preprocessing
scaler_after  = StandardScaler()
X_after       = scaler_after.fit_transform(spectra_pp)
pca_after     = PCA(n_components=2, random_state=42)
pc_after      = pca_after.fit_transform(X_after)

print(f"PCA before preprocessing:")
print(f"  PC1 explains: {pca_before.explained_variance_ratio_[0]*100:.1f}%")
print(f"  PC2 explains: {pca_before.explained_variance_ratio_[1]*100:.1f}%")
print(f"\nPCA after preprocessing:")
print(f"  PC1 explains: {pca_after.explained_variance_ratio_[0]*100:.1f}%")
print(f"  PC2 explains: {pca_after.explained_variance_ratio_[1]*100:.1f}%")


In [ ]:
# Colour by TIC to reveal spatial structure
tic_normalised = tic / tic.max()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, pc, title, exp in zip(
    axes,
    [pc_before, pc_after],
    ["PCA — before preprocessing", "PCA — after preprocessing"],
    [pca_before.explained_variance_ratio_,
     pca_after.explained_variance_ratio_]
):
    sc = ax.scatter(pc[:, 0], pc[:, 1],
                    c=tic_normalised, cmap="viridis",
                    s=1, alpha=0.5)
    plt.colorbar(sc, ax=ax, label="Normalised TIC", shrink=0.8)
    ax.set_xlabel(f"PC1 ({exp[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({exp[1]*100:.1f}%)")
    ax.set_title(title, fontsize=10)

fig.suptitle("PCA of pixel spectra — coloured by TIC\n"
             "Distinct groups suggest molecular tissue regions",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/01_pca_before_after.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


## 10. t-SNE — After Preprocessing

t-SNE (t-distributed Stochastic Neighbour Embedding) is a
non-linear dimensionality reduction method that preserves
local structure better than PCA. It is particularly useful
for revealing cluster structure in high-dimensional data.

Unlike PCA, t-SNE:
- Preserves local neighbourhoods (nearby pixels in m/z space stay nearby)
- Can reveal non-linear structure
- Is better at separating distinct groups
- But: axes have no direct biological meaning

We run t-SNE on the preprocessed data and colour by spatial location
to understand whether spatial proximity correlates with molecular similarity.

Reference: van der Maaten & Hinton (2008) *J. Mach. Learn. Res.* 9:2579


In [ ]:
from sklearn.manifold import TSNE

print("Running t-SNE (this may take a few minutes)...")

tsne   = TSNE(n_components=2, random_state=42,
              perplexity=30, n_iter=1000, verbose=1)
X_tsne = tsne.fit_transform(X_after)

print(f"t-SNE complete. Shape: {X_tsne.shape}")


In [ ]:
# Colour by x and y coordinates separately to reveal spatial structure
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Colour by x coordinate
sc0 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1],
                       c=coordinates["x"], cmap="RdYlBu",
                       s=1, alpha=0.5)
plt.colorbar(sc0, ax=axes[0], label="x coordinate", shrink=0.8)
axes[0].set_xlabel("t-SNE 1")
axes[0].set_ylabel("t-SNE 2")
axes[0].set_title("t-SNE coloured by x position", fontsize=10)

# Colour by y coordinate
sc1 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                       c=coordinates["y"], cmap="RdYlBu",
                       s=1, alpha=0.5)
plt.colorbar(sc1, ax=axes[1], label="y coordinate", shrink=0.8)
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")
axes[1].set_title("t-SNE coloured by y position", fontsize=10)

fig.suptitle("t-SNE of preprocessed pixel spectra\n"
             "Spatial gradients reveal tissue organisation",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/01_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


## 11. Save Preprocessed Data

Save the preprocessed matrix and coordinates for use in notebooks 02 and 03.


In [ ]:
np.save("results/spectra_preprocessed.npy", spectra_pp)
np.save("results/spectra_norm.npy",         spectra_norm)
np.save("results/reference_mz.npy",         ref_mz)
np.save("results/mz_values.npy",            mz_values)
np.save("results/pca_embedding.npy",        pc_after)
np.save("results/tsne_embedding.npy",       X_tsne)
coordinates.to_csv("results/coordinates.csv", index=False)

print("Saved:")
print("  results/spectra_preprocessed.npy")
print("  results/spectra_norm.npy")
print("  results/reference_mz.npy")
print("  results/pca_embedding.npy")
print("  results/tsne_embedding.npy")
print("  results/coordinates.csv")


## Summary

| Step | What we did | Key finding |
|---|---|---|
| Data loading | Loaded {n_pixels} pixels × {n_mz} m/z values | Centroided data, 400-1000 Da |
| TIC map | Spatial overview of molecular signal | Tissue outline clearly visible |
| Ion images | Top 9 molecular distributions | Different molecules in different regions |
| Pixel comparison | Centre vs periphery spectra | Distinct molecular profiles |
| Preprocessing | TIC norm → 2.5% filter → 10 ppm rebin | Reduced to reference peak matrix |
| Top 5 peaks | Most abundant molecules with lipid class | Predominantly PC/SM lipids |
| PCA | Variance structure before/after preprocessing | Groups emerge after preprocessing |
| t-SNE | Non-linear structure in pixel space | Spatial gradients visible |

**Next:** Notebook 02 — Segmentation using K-means, GMM, and Spectral Clustering
